In [ ]:
!pip install ultralytics
!pip install ipywidgets
!pip install pillow
!pip install matplotlib
!pip install opencv-python

!jupyter nbextension enable --py widgetsnbextension --sys-prefix

print("All packages installed successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 11.6 MB/s eta 0:00:00
Enabling notebook extension jupyter-js-widgets/extension...
      - Validating: OK
All packages installed successfully!


In [ ]:
import io
import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import ipywidgets as widgets
from IPython.display import display, clear_output
from ultralytics import YOLO
import warnings
warnings.filterwarnings('ignore')
print("Libraries imported successfully!")

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Libraries imported successfully!


In [ ]:
print("Loading YOLOv8 model...")
model = YOLO('yolov8n.pt')

print(f"Model loaded successfully!")
print(f"Model can detect {len(model.names)} different classes:")
print(list(model.names.values())[:10], "... and more")

Loading YOLOv8 model...
Model loaded successfully!
Model can detect 80 different classes:
['person', 'bicycle', 'car', 'motorcycle', 'airplane', 'bus', 'train', 'truck', 'boat', 'traffic light'] ... and more


In [ ]:
def detect_objects(image_data, confidence_threshold=0.25):
    """
    Perform object detection on uploaded image

    Args:
        image_data: Image data from file upload
        confidence_threshold: Minimum confidence for detections
    """
    try:

        image = Image.open(io.BytesIO(image_data))

        if image.mode != 'RGB':
            image = image.convert('RGB')

        img_array = np.array(image)

        print(f"Image loaded: {image.size[0]}x{image.size[1]} pixels")

        print("Performing object detection...")
        results = model.predict(img_array, conf=confidence_threshold, verbose=False)

        result = results[0]

        annotated_image = result.plot()

        annotated_image_rgb = cv2.cvtColor(annotated_image, cv2.COLOR_BGR2RGB)

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 10))

        ax1.imshow(img_array)
        ax1.set_title('Original Image', fontsize=16)
        ax1.axis('off')

        ax2.imshow(annotated_image_rgb)
        ax2.set_title(f'Object Detection Results (Confidence ≥ {confidence_threshold})', fontsize=16)
        ax2.axis('off')

        plt.tight_layout()
        plt.show()

        if len(result.boxes) > 0:
            print(f"\n🎯 Detection Results:")
            print(f"Found {len(result.boxes)} objects:\n")

            for i, box in enumerate(result.boxes):
                class_id = int(box.cls[0])
                class_name = model.names[class_id]
                confidence = float(box.conf[0])

                x1, y1, x2, y2 = box.xyxy[0].tolist()

                print(f"{i+1}. {class_name.upper()} - Confidence: {confidence:.2%}")
                print(f"   Location: ({int(x1)}, {int(y1)}) to ({int(x2)}, {int(y2)})")
                print()
        else:
            print(f"No objects detected with confidence ≥ {confidence_threshold}")
            print("Try lowering the confidence threshold or upload a different image.")

    except Exception as e:
        print(f"Error processing image: {str(e)}")
        print("Please make sure you uploaded a valid JPG or PNG image.")

print("Object detection function created successfully!")

Object detection function created successfully!


In [ ]:

file_upload = widgets.FileUpload(
    accept='image/*,.jpg,.jpeg,.png',
    multiple=False,
    description='Upload Image'
)
confidence_slider = widgets.FloatSlider(
    value=0.25,
    min=0.1,
    max=0.9,
    step=0.05,
    description='Confidence:',
    style={'description_width': 'initial'}
)
detect_button = widgets.Button(
    description='Detect Objects',
    button_style='primary',
    layout=widgets.Layout(width='200px', height='40px')
)
output = widgets.Output()

def on_detect_click(b):
    """Handle detect button click"""
    with output:
        clear_output(wait=True)

        if not file_upload.value:
            print(" Please upload an image first!")
            return
        uploaded_file = list(file_upload.value.values())[0]
        file_content = uploaded_file['content']
        file_name = uploaded_file['metadata']['name']

        print(f"Processing: {file_name}")
        print(f"Confidence threshold: {confidence_slider.value:.2%}")
        print("" + "="*50)
        detect_objects(file_content, confidence_slider.value)
detect_button.on_click(on_detect_click)

print("File upload widget created successfully!")

File upload widget created successfully!


In [ ]:
print("YOLOv8 Object Detection Interface")
print("Upload an image and click 'Detect Objects' to see the results!\n")
interface = widgets.VBox([
    widgets.HTML("<h3>Upload Image</h3>"),
    file_upload,
    widgets.HTML("<h3>Settings</h3>"),
    confidence_slider,
    widgets.HTML("<h3>Detection</h3>"),
    detect_button,
    widgets.HTML("<h3> Results</h3>"),
    output
])

display(interface)

YOLOv8 Object Detection Interface
Upload an image and click 'Detect Objects' to see the results!



In [ ]:
def batch_detect_from_urls(image_urls, confidence_threshold=0.25):
    """
    Detect objects in multiple images from URLs

    Args:
        image_urls: List of image URLs
        confidence_threshold: Minimum confidence for detections
    """
    for i, url in enumerate(image_urls):
        print(f"\n{'='*60}")
        print(f"Processing Image {i+1}/{len(image_urls)}: {url}")
        print(f"{'='*60}")

        try:
            results = model.predict(url, conf=confidence_threshold, verbose=False)
            result = results[0]

            annotated_image = result.plot()
            annotated_image_rgb = cv2.cvtColor(annotated_image, cv2.COLOR_BGR2RGB)

            plt.figure(figsize=(12, 8))
            plt.imshow(annotated_image_rgb)
            plt.title(f'Detection Results - Image {i+1}', fontsize=16)
            plt.axis('off')
            plt.show()
            if len(result.boxes) > 0:
                detected_classes = [model.names[int(box.cls[0])] for box in result.boxes]
                class_counts = {cls: detected_classes.count(cls) for cls in set(detected_classes)}

                print(f"Found {len(result.boxes)} objects:")
                for cls, count in class_counts.items():
                    print(f"  - {cls}: {count}")
            else:
                print("No objects detected")

        except Exception as e:
            print(f"Error processing {url}: {str(e)}")

print("Batch processing function ready! Uncomment the example above to test.")

Batch processing function ready! Uncomment the example above to test.


In [ ]:
# Display all detectable classes
print("All Detectable Object Classes:\n")

classes = list(model.names.values())

# Display classes in columns
for i in range(0, len(classes), 4):
    row = classes[i:i+4]
    print(f"{i//4 + 1:2d}. {row[0]:<15} {row[1] if len(row) > 1 else '':<15} {row[2] if len(row) > 2 else '':<15} {row[3] if len(row) > 3 else '':<15}")

print(f"\nTotal: {len(classes)} classes")

All Detectable Object Classes:

 1. person          bicycle         car             motorcycle     
 2. airplane        bus             train           truck          
 3. boat            traffic light   fire hydrant    stop sign      
 4. parking meter   bench           bird            cat            
 5. dog             horse           sheep           cow            
 6. elephant        bear            zebra           giraffe        
 7. backpack        umbrella        handbag         tie            
 8. suitcase        frisbee         skis            snowboard      
 9. sports ball     kite            baseball bat    baseball glove 
10. skateboard      surfboard       tennis racket   bottle         
11. wine glass      cup             fork            knife          
12. spoon           bowl            banana          apple          
13. sandwich        orange          broccoli        carrot         
14. hot dog         pizza           donut           cake           
15. chair       